In [1]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/sidecar_manager.py
import json
from pathlib import Path

class SidecarManager:
    def __init__(self, filepath="sidecar_edits.json"):
        # Inizializzazione di default; modificabile passandogli un diverso path come parametro
        self.filepath = Path(filepath) if filepath else Path("sidecar_edits.json")
        self._ensure_file_exists()

    def _ensure_file_exists(self):
        self.filepath.parent.mkdir(parents=True, exist_ok=True)
        if not self.filepath.exists():
            with open(self.filepath, "w", encoding="utf-8") as f:
                json.dump({"pairwise_deltas": {}, "tag_overrides": {}}, f, indent=2)

    @property
    def data(self) -> dict:
        """Garantisce l'accesso diretto ai dati leggendoli sempre aggiornati dal file."""
        return self.load_data()

    def load_data(self) -> dict:
        if self.filepath.exists():
            try:
                with open(self.filepath, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                return {"pairwise_deltas": {}, "tag_overrides": {}}
        return {"pairwise_deltas": {}, "tag_overrides": {}}

    def save_pairwise_delta(self, chunk_id_1: str, chunk_id_2: str, distance_factor: float):
        """
        Salva o aggiorna il fattore di distanza tra una coppia di chunk.
        Garantisce la simmetria della relazione (A_B == B_A).
        """
        data = self.load_data()
        
        # Ordiniamo gli ID per garantire che la relazione sia simmetrica
        pair_key = "_AND_".join(sorted([str(chunk_id_1), str(chunk_id_2)]))
        
        if "pairwise_deltas" not in data:
            data["pairwise_deltas"] = {}

        data["pairwise_deltas"][pair_key] = {
            "chunk_1": str(chunk_id_1),
            "chunk_2": str(chunk_id_2),
            "distance_factor": round(distance_factor, 3)
        }

        with open(self.filepath, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        print(f"<<| Modifica salvata per la coppia [{pair_key}]: factor={distance_factor:.2f} |>>")

    def reset_all(self):
        """Ripristina il file sidecar azzerando le modifiche (UNDO globale)."""
        with open(self.filepath, "w", encoding="utf-8") as f:
            json.dump({"pairwise_deltas": {}, "tag_overrides": {}}, f, indent=2)
EOF

In [1]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/frontend/chunk_graph.js
import * as d3 from "https://esm.sh/d3@7";

export function render({ model, el }) {
  el.innerHTML = "";

  const container = d3.select(el)
    .append("div")
    .style("position", "relative")
    .style("width", "650px")
    .style("font-family", "sans-serif");

  const width = 650;
  const height = 420;

  const svg = container.append("svg")
    .attr("width", width)
    .attr("height", height)
    .style("background", "#f8fafc")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "8px");

  const infoBox = container.append("div")
    .style("position", "absolute")
    .style("top", "12px")
    .style("right", "12px")
    .style("width", "220px")
    .style("padding", "10px")
    .style("background", "rgba(255, 255, 255, 0.95)")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "6px")
    .style("font-size", "12px")
    .style("color", "#334155")
    .style("pointer-events", "none")
    .html("<b>📌 Info Chunk</b><br><span style='color:#94a3b8;'>Clicca un nodo per vederne il testo</span>");

  function draw() {
    const graph = model.get("graph_data");
    if (!graph || !graph.nodes || graph.nodes.length === 0) return;

    svg.selectAll("*").remove();

    const nodes = graph.nodes.map(d => ({ ...d }));
    const links = graph.links ? graph.links.map(d => ({ ...d })) : [];

    const BASE_DISTANCE = 120;

    const simulation = d3.forceSimulation(nodes)
      .force("link", d3.forceLink(links)
        .id(d => d.id)
        .distance(d => BASE_DISTANCE * (d.distance_factor || 1.0))
      )
      .force("charge", d3.forceManyBody().strength(-180))
      .force("center", d3.forceCenter(width / 2, height / 2));

    const link = svg.append("g")
      .selectAll("line")
      .data(links)
      .enter().append("line")
      .attr("stroke", "#94a3b8")
      .attr("stroke-width", 2);

    const linkText = svg.append("g")
      .selectAll("text")
      .data(links)
      .enter().append("text")
      .attr("font-size", "11px")
      .attr("font-weight", "bold")
      .attr("fill", "#0284c7")
      .attr("text-anchor", "middle");

    const node = svg.append("g")
      .selectAll("circle")
      .data(nodes)
      .enter().append("circle")
      .attr("r", 12)
      .attr("fill", "#6366f1")
      .attr("stroke", "#ffffff")
      .attr("stroke-width", 2)
      .style("cursor", "grab");

    const label = svg.append("g")
      .selectAll("text")
      .data(nodes)
      .enter().append("text")
      .text(d => d.id)
      .attr("font-size", "11px")
      .attr("dx", 15)
      .attr("dy", 4)
      .attr("fill", "#1e293b");

    // Blocca permanentemente TUTTI i nodi appena il layout iniziale è pronto
    simulation.on("end", () => {
      nodes.forEach(n => {
        n.fx = n.x;
        n.fy = n.y;
      });
    });

    const drag = d3.drag()
      .on("start", (event, d) => {
        // Congela istantaneamente la posizione di tutti i nodi
        nodes.forEach(n => {
          n.fx = n.x;
          n.fy = n.y;
        });
      })
      .on("drag", (event, d) => {
        // Aggiorna solo il nodo trascinato senza svegliare la fisica di D3
        d.fx = event.x;
        d.fy = event.y;
        d.x = event.x;
        d.y = event.y;
        updatePositions();
      })
      .on("end", (event, d) => {
        d.fx = event.x;
        d.fy = event.y;
        d.x = event.x;
        d.y = event.y;

        links.forEach(l => {
          const srcId = typeof l.source === 'object' ? l.source.id : l.source;
          const tgtId = typeof l.target === 'object' ? l.target.id : l.target;

          if (srcId === d.id || tgtId === d.id) {
            const dx = l.target.x - l.source.x;
            const dy = l.target.y - l.source.y;
            const currentDist = Math.sqrt(dx * dx + dy * dy);
            const factor = currentDist / BASE_DISTANCE;

            l.distance_factor = factor;

            model.set("pairwise_edit", {
              chunk_1: srcId,
              chunk_2: tgtId,
              distance_factor: factor
            });
            model.save_changes();
          }
        });

        updatePositions();
      });

    node.call(drag);

    node.on("click", (event, d) => {
      node.attr("fill", n => n.id === d.id ? "#ef4444" : "#6366f1");
      infoBox.html(`<b>🆔 ${d.id}</b><br><span style="color:#334155;">📖 ${d.text || "Nessun testo"}</span>`);
      model.set("selected_tag", { id: d.id, text: d.text || "" });
      model.save_changes();
    });

    function updatePositions() {
      link
        .attr("x1", d => d.source.x)
        .attr("y1", d => d.source.y)
        .attr("x2", d => d.target.x)
        .attr("y2", d => d.target.y);

      linkText
        .attr("x", d => (d.source.x + d.target.x) / 2)
        .attr("y", d => (d.source.y + d.target.y) / 2 - 5)
        .text(d => {
          const dx = d.target.x - d.source.x;
          const dy = d.target.y - d.source.y;
          const dist = Math.round(Math.sqrt(dx * dx + dy * dy));
          const factor = (dist / BASE_DISTANCE).toFixed(2);
          return `${dist}px (${factor}x)`;
        });

      node
        .attr("cx", d => d.x)
        .attr("cy", d => d.y);

      label
        .attr("x", d => d.x)
        .attr("y", d => d.y);
    }

    simulation.on("tick", updatePositions);
  }

  model.on("change:graph_data", draw);
  draw();
}
EOF

In [ ]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/chunk_widget.py
import pathlib
import anywidget
import traitlets
from sidecar_manager import SidecarManager

class ChunkGraphWidget(anywidget.AnyWidget):
    _esm = pathlib.Path(__file__).parent / "frontend" / "chunk_graph.js"

    graph_data = traitlets.Dict({"nodes": [], "links": []}).tag(sync=True)
    selected_tag = traitlets.Dict({}).tag(sync=True)
    pairwise_edit = traitlets.Dict({}).tag(sync=True)

    def __init__(self, sidecar_path=None, **kwargs):
        super().__init__(**kwargs)
        self.sidecar = SidecarManager(filepath=sidecar_path) if sidecar_path else SidecarManager()
        self.observe(self._on_pairwise_edit, names=["pairwise_edit"])

    def load_graph(self, raw_graph_data):
        """Carica il grafo applicando immediatamente i distance_factor salvati nel sidecar."""
        # Recupero sicuro dei delta
        sidecar_dict = getattr(self.sidecar, "data", {})
        deltas = sidecar_dict.get("pairwise_deltas", {}) if isinstance(sidecar_dict, dict) else {}
        
        links = raw_graph_data.get("links", [])
        enriched_links = []

        for link in links:
            l_copy = dict(link)
            src = l_copy["source"]["id"] if isinstance(l_copy["source"], dict) else l_copy["source"]
            tgt = l_copy["target"]["id"] if isinstance(l_copy["target"], dict) else l_copy["target"]

            k1 = f"{src}_AND_{tgt}"
            k2 = f"{tgt}_AND_{src}"

            factor = 1.0
            if k1 in deltas:
                factor = deltas[k1].get("distance_factor", 1.0)
            elif k2 in deltas:
                factor = deltas[k2].get("distance_factor", 1.0)

            l_copy["distance_factor"] = factor
            enriched_links.append(l_copy)

        self.graph_data = {
            "nodes": raw_graph_data.get("nodes", []),
            "links": enriched_links
        }

    def _on_pairwise_edit(self, change):
        edit = change["new"]
        if edit and "chunk_1" in edit and "chunk_2" in edit:
            self.sidecar.save_pairwise_delta(
                chunk_id_1=edit["chunk_1"],
                chunk_id_2=edit["chunk_2"],
                distance_factor=edit["distance_factor"]
            )
EOF